# 09 - AgentCore Policy
Create a Policy Engine, author Cedar policies for tool access control, attach policies to the gateway, and test allow/deny scenarios.

## Step 1: Load Configuration

In [ ]:
import boto3
import json
import time
from pathlib import Path
from cleanup_utils import ResourceTracker

config = json.loads(Path("config.json").read_text())
REGION = config["REGION"]
PREFIX = config["PROJECT_PREFIX"]
GATEWAY_ID = config["GATEWAY_ID"]

tracker = ResourceTracker()
agentcore = boto3.client("bedrock-agentcore", region_name=REGION)

print(f"\u2713 Config loaded (region={REGION}, gateway={GATEWAY_ID})")

## Step 2: Create Policy Engine

The Policy Engine evaluates Cedar policies to control agent access to tools.

In [ ]:
POLICY_ENGINE_NAME = f"{PREFIX}-policy-engine"

# Check if policy engine exists
existing_engines = agentcore.list_policy_engines()
existing = next(
    (e for e in existing_engines.get("policyEngines", []) if e["name"] == POLICY_ENGINE_NAME),
    None
)

if existing:
    policy_engine_id = existing["id"]
    print(f"\u2713 Policy engine already exists: {policy_engine_id}")
else:
    print(f"\u231b Creating policy engine: {POLICY_ENGINE_NAME}...")
    response = agentcore.create_policy_engine(
        name=POLICY_ENGINE_NAME,
        description="OrgIQ access control engine using Cedar policies",
        tags={"project": PREFIX}
    )
    policy_engine_id = response["id"]
    print(f"\u2713 Policy engine created: {policy_engine_id}")

## Step 3: Define Cedar Schema

Cedar requires a schema that defines entity types, actions, and their relationships.

In [ ]:
# Cedar schema for OrgIQ access control
cedar_schema = {
    "OrgIQ": {
        "entityTypes": {
            "Agent": {
                "shape": {
                    "type": "Record",
                    "attributes": {
                        "name": {"type": "String"},
                        "tier": {"type": "String"}
                    }
                },
                "memberOfTypes": ["AgentGroup"]
            },
            "AgentGroup": {
                "shape": {
                    "type": "Record",
                    "attributes": {
                        "name": {"type": "String"}
                    }
                }
            },
            "Tool": {
                "shape": {
                    "type": "Record",
                    "attributes": {
                        "name": {"type": "String"},
                        "sensitivity": {"type": "String"}
                    }
                },
                "memberOfTypes": ["ToolGroup"]
            },
            "ToolGroup": {
                "shape": {
                    "type": "Record",
                    "attributes": {
                        "name": {"type": "String"}
                    }
                }
            }
        },
        "actions": {
            "invoke": {
                "appliesTo": {
                    "principalTypes": ["Agent", "AgentGroup"],
                    "resourceTypes": ["Tool", "ToolGroup"]
                }
            },
            "discover": {
                "appliesTo": {
                    "principalTypes": ["Agent", "AgentGroup"],
                    "resourceTypes": ["Tool", "ToolGroup"]
                }
            }
        }
    }
}

# Upload schema to policy engine
agentcore.put_policy_engine_schema(
    policyEngineId=policy_engine_id,
    schema=json.dumps(cedar_schema)
)

print("\u2713 Cedar schema uploaded")
print(f"  Entity types: Agent, AgentGroup, Tool, ToolGroup")
print(f"  Actions: invoke, discover")

## Step 4: Create Cedar Policies

Define access control policies using Cedar policy language.

In [ ]:
# Policy 1: Allow all agents to discover tools
discover_policy = """
permit(
    principal,
    action == OrgIQ::Action::"discover",
    resource
)
when { true };
"""

agentcore.create_policy(
    policyEngineId=policy_engine_id,
    name="allow-discovery",
    description="All agents can discover available tools",
    policyBody=discover_policy
)
print("\u2713 Policy created: allow-discovery")

In [ ]:
# Policy 2: Allow search agents to invoke read-only tools
search_invoke_policy = """
permit(
    principal == OrgIQ::Agent::"orgiq-search",
    action == OrgIQ::Action::"invoke",
    resource
)
when { resource.sensitivity == "low" || resource.sensitivity == "medium" };
"""

agentcore.create_policy(
    policyEngineId=policy_engine_id,
    name="search-agent-invoke",
    description="Search agent can invoke low and medium sensitivity tools",
    policyBody=search_invoke_policy
)
print("\u2713 Policy created: search-agent-invoke")

In [ ]:
# Policy 3: Allow supervisor to invoke all tools
supervisor_policy = """
permit(
    principal == OrgIQ::Agent::"orgiq-supervisor",
    action == OrgIQ::Action::"invoke",
    resource
);
"""

agentcore.create_policy(
    policyEngineId=policy_engine_id,
    name="supervisor-full-access",
    description="Supervisor agent has full tool access",
    policyBody=supervisor_policy
)
print("\u2713 Policy created: supervisor-full-access")

In [ ]:
# Policy 4: Deny all agents from invoking high-sensitivity tools by default
deny_high_policy = """
forbid(
    principal,
    action == OrgIQ::Action::"invoke",
    resource
)
when { resource.sensitivity == "high" }
unless { principal == OrgIQ::Agent::"orgiq-supervisor" };
"""

agentcore.create_policy(
    policyEngineId=policy_engine_id,
    name="deny-high-sensitivity",
    description="Block high sensitivity tools unless supervisor",
    policyBody=deny_high_policy
)
print("\u2713 Policy created: deny-high-sensitivity")

## Step 5: Attach Policy Engine to Gateway

In [ ]:
# Attach the policy engine to the gateway
agentcore.update_gateway(
    gatewayId=GATEWAY_ID,
    policyEngineId=policy_engine_id
)

print(f"\u2713 Policy engine attached to gateway")
print(f"  Gateway: {GATEWAY_ID}")
print(f"  Policy Engine: {policy_engine_id}")

## Step 6: Test Authorization - ALLOW Scenario

In [ ]:
# Test: search agent invoking a low-sensitivity tool (should ALLOW)
auth_response = agentcore.check_authorization(
    policyEngineId=policy_engine_id,
    principal={
        "entityType": "OrgIQ::Agent",
        "entityId": "orgiq-search"
    },
    action={
        "actionType": "OrgIQ::Action",
        "actionId": "invoke"
    },
    resource={
        "entityType": "OrgIQ::Tool",
        "entityId": "graph-query",
        "attributes": {"name": "graph-query", "sensitivity": "low"}
    }
)

decision = auth_response["decision"]
print(f"Test: search agent -> invoke -> low-sensitivity tool")
print(f"  Decision: {decision}")
assert decision == "ALLOW", f"Expected ALLOW, got {decision}"
print(f"  \u2713 Correctly ALLOWED")

## Step 7: Test Authorization - DENY Scenario

In [ ]:
# Test: search agent invoking a high-sensitivity tool (should DENY)
auth_response = agentcore.check_authorization(
    policyEngineId=policy_engine_id,
    principal={
        "entityType": "OrgIQ::Agent",
        "entityId": "orgiq-search"
    },
    action={
        "actionType": "OrgIQ::Action",
        "actionId": "invoke"
    },
    resource={
        "entityType": "OrgIQ::Tool",
        "entityId": "admin-delete",
        "attributes": {"name": "admin-delete", "sensitivity": "high"}
    }
)

decision = auth_response["decision"]
print(f"Test: search agent -> invoke -> high-sensitivity tool")
print(f"  Decision: {decision}")
assert decision == "DENY", f"Expected DENY, got {decision}"
print(f"  \u2713 Correctly DENIED")

## Step 8: Test Supervisor Override

In [ ]:
# Test: supervisor agent invoking a high-sensitivity tool (should ALLOW)
auth_response = agentcore.check_authorization(
    policyEngineId=policy_engine_id,
    principal={
        "entityType": "OrgIQ::Agent",
        "entityId": "orgiq-supervisor"
    },
    action={
        "actionType": "OrgIQ::Action",
        "actionId": "invoke"
    },
    resource={
        "entityType": "OrgIQ::Tool",
        "entityId": "admin-delete",
        "attributes": {"name": "admin-delete", "sensitivity": "high"}
    }
)

decision = auth_response["decision"]
print(f"Test: supervisor agent -> invoke -> high-sensitivity tool")
print(f"  Decision: {decision}")
assert decision == "ALLOW", f"Expected ALLOW, got {decision}"
print(f"  \u2713 Supervisor override working")

## Step 9: Test Discovery (Universal Allow)

In [ ]:
# Test: any agent can discover tools
auth_response = agentcore.check_authorization(
    policyEngineId=policy_engine_id,
    principal={
        "entityType": "OrgIQ::Agent",
        "entityId": "orgiq-unknown-agent"
    },
    action={
        "actionType": "OrgIQ::Action",
        "actionId": "discover"
    },
    resource={
        "entityType": "OrgIQ::Tool",
        "entityId": "admin-delete",
        "attributes": {"name": "admin-delete", "sensitivity": "high"}
    }
)

decision = auth_response["decision"]
print(f"Test: unknown agent -> discover -> any tool")
print(f"  Decision: {decision}")
assert decision == "ALLOW", f"Expected ALLOW, got {decision}"
print(f"  \u2713 Universal discovery working")

## Step 10: List Policies

In [ ]:
# List all policies in the engine
policies = agentcore.list_policies(policyEngineId=policy_engine_id)

print(f"Policies in engine ({len(policies.get('policies', []))}):\n")
for policy in policies.get("policies", []):
    print(f"  {policy['name']}")
    print(f"    {policy.get('description', '')}")
    print()

## Step 11: Save Configuration

In [ ]:
config["POLICY_ENGINE_ID"] = policy_engine_id
Path("config.json").write_text(json.dumps(config, indent=2))

print(f"\u2713 Config saved")
print(f"  POLICY_ENGINE_ID: {policy_engine_id}")

## Summary

- [x] Policy Engine created
- [x] Cedar schema defined (Agent, Tool, AgentGroup, ToolGroup entities)
- [x] Four Cedar policies authored (discover, search-invoke, supervisor, deny-high)
- [x] Policy Engine attached to MCP Gateway
- [x] ALLOW scenario tested (search agent + low-sensitivity tool)
- [x] DENY scenario tested (search agent + high-sensitivity tool)
- [x] Supervisor override tested (supervisor + high-sensitivity tool)
- [x] Universal discovery permission tested

**Next:** [10-agentcore-identity.ipynb](10-agentcore-identity.ipynb) — Create workload identities and OAuth credentials